# LangChain: Evaluation (Groq Llama 3.1)

## Outline
* **Example Generation** - Create test data (manual + LLM-generated)
* **Manual Evaluation** - Debug mode to inspect chain internals
* **LLM-Assisted Evaluation** - Use LLM to grade responses
* **Evaluation Platform** - Track and visualize results

**Model Used:** Groq Llama-3.1-8b-instant  
**Key Concept:** Evaluating LLM applications is challenging because outputs are open-ended text, not exact matches.

**Why This Matters:** You need to know if your changes improve or degrade your application's performance.


In [1]:
# Cell 2: Install Required Packages

!pip install langchain langchain-groq python-dotenv docarray sentence-transformers -q


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Cell 3: Import Libraries and Load Environment

import os
import warnings
from dotenv import load_dotenv, find_dotenv

# Load environment variables
load_dotenv(find_dotenv())

# Suppress warnings
warnings.filterwarnings('ignore')

# Verify API key
groq_api_key = os.environ.get('GROQ_API_KEY')
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in .env file!")
    
print("✅ Environment loaded successfully")


✅ Environment loaded successfully


In [5]:
# Cell 4: Create Q&A Application

from langchain_community.document_loaders import CSVLoader
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Load data
file = 'OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file, encoding='utf-8')
data = loader.load()

print(f"✅ Loaded {len(data)} documents from {file}")

# Create vectorstore
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = DocArrayInMemorySearch.from_documents(data, embedding=embeddings)
retriever = vectorstore.as_retriever()

print("✅ Vector store created")

# Initialize LLM
llm_model = "llama-3.1-8b-instant"
llm = ChatGroq(temperature=0.0, model=llm_model, groq_api_key=groq_api_key)

# Create prompt
prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:

{context}

Question: {question}
""")

# Create QA chain
qa = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ QA chain created successfully")

✅ Loaded 1000 documents from OutdoorClothingCatalog_1000.csv


C:\Users\Client\AppData\Local\Temp\ipykernel_34320\467234903.py:19: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Vector store created
✅ QA chain created successfully


In [7]:
# Cell 5: Create Vector Store Index

# Vector store already created in cell 4

print("✅ Vector store index created")

✅ Vector store index created


In [8]:
# Cell 6: Initialize LLM and QA Chain

# Already initialized in cell 4

print("✅ QA chain created successfully")

✅ QA chain created successfully


In [9]:
# Cell 7: Inspect Sample Documents

# Look at some sample documents to understand the data
print("Document 10:")
print(data[10])
print("\n" + "="*60 + "\n")

print("Document 11:")
print(data[11])


Document 10:
page_content=': 10
name: Cozy Comfort Pullover Set, Stripe
description: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.

Size & Fit
- Pants are Favorite Fit: Sits lower on the waist.
- Relaxed Fit: Our most generous fit sits farthest from the body.

Fabric & Care
- In the softest blend of 63% polyester, 35% rayon and 2% spandex.

Additional Features
- Relaxed fit top with raglan sleeves and rounded hem.
- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.

Imported.' metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 10}


Document 11:
page_content=': 11
name: Ultra-Lofty 850 Stretch Down Hooded Jacket
description: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range

## Coming Up With Test Data

We need examples to evaluate our QA system. There are two approaches:
1. **Manual** - Create examples by hand (time-consuming but precise)
2. **LLM-Generated** - Use an LLM to generate examples (fast but needs review)


In [10]:
# Cell 8: Create Hard-Coded Examples

# Manually created question-answer pairs based on the documents above
examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

print(f"✅ Created {len(examples)} manual examples")


✅ Created 2 manual examples


In [ ]:
# Cell 9: LLM-Generated Examples

from langchain.evaluation.qa import QAGenerateChain

# Create a chain that generates Q&A pairs from documents
example_gen_chain = QAGenerateChain.from_llm(
    ChatGroq(model=llm_model, groq_api_key=groq_api_key)
)

print("✅ QA generation chain created")


In [11]:
# Cell 10: Generate Examples from Documents

from langchain_core.prompts import ChatPromptTemplate

# Create a prompt for generating Q&A pairs
qa_prompt = ChatPromptTemplate.from_template("""Generate a question and answer pair based on the following document. The question should be answerable from the document, and the answer should be concise.

Document: {doc}

Format your response as:
Question: <question>
Answer: <answer>
""")

# Create generation chain
qa_gen_chain = qa_prompt | llm | StrOutputParser()

# Generate Q&A pairs from first 5 documents
new_examples = []
for doc in data[:5]:
    response = qa_gen_chain.invoke({"doc": doc.page_content})
    # Parse the response
    lines = response.split('\n')
    question = ""
    answer = ""
    for line in lines:
        if line.startswith("Question:"):
            question = line.replace("Question:", "").strip()
        elif line.startswith("Answer:"):
            answer = line.replace("Answer:", "").strip()
    if question and answer:
        new_examples.append({"query": question, "answer": answer})

print(f"✅ Generated {len(new_examples)} new examples")
print("\nFirst generated example:")
print(new_examples[0])

✅ Generated 5 new examples

First generated example:
{'query': "What material is used for the innersole of the Women's Campside Oxfords?", 'answer': 'EVA (with Cleansport NXT antimicrobial odor control)'}


In [12]:
# Cell 11: Compare Generated Example with Source

print("Generated Question:")
print(new_examples[0]['query'])
print("\nGenerated Answer:")
print(new_examples[0]['answer'])
print("\n" + "="*60)
print("\nSource Document:")
print(data[0])


Generated Question:
What material is used for the innersole of the Women's Campside Oxfords?

Generated Answer:
EVA (with Cleansport NXT antimicrobial odor control)


Source Document:
page_content=': 0
name: Women's Campside Oxfords
description: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. 

Size & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. 

Specs: Approx. weight: 1 lb.1 oz. per pair. 

Construction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. 

Questions? Please contact us for any inquiries.' metadata={'source': 'Outdoor

In [13]:
# Cell 12: Combine All Examples

# Combine manual and LLM-generated examples
examples += new_examples

print(f"✅ Total examples: {len(examples)}")
print(f"   - Manual: 2")
print(f"   - LLM-generated: {len(new_examples)}")


✅ Total examples: 7
   - Manual: 2
   - LLM-generated: 5


In [14]:
# Cell 13: Test QA Chain on First Example

# Run the first example through the chain
result = qa.invoke(examples[0]["query"])

print("Query:")
print(examples[0]["query"])
print("\nExpected Answer:")
print(examples[0]["answer"])
print("\nActual Answer:")
print(result)

Query:
Do the Cozy Comfort Pullover Set have side pockets?

Expected Answer:
Yes

Actual Answer:
Yes, the Cozy Comfort Pullover Set has side pockets. This information can be found in the "Additional Features" section of the document with metadata {'source': 'OutdoorClothingCatalog_1000.csv', 'row': 10}.


## Manual Evaluation

To understand what's happening inside the chain, we can enable **debug mode**. This shows:
- The exact prompt sent to the LLM
- Retrieved documents used as context
- Intermediate steps in multi-step chains
- Token usage and costs


In [15]:
# Cell 14: Enable Debug Mode

import langchain
langchain.debug = True

print("✅ Debug mode enabled")


✅ Debug mode enabled


In [16]:
# Cell 17: Run Example with Debug Output

# Run the same example with debug mode on
result = qa.invoke(examples[0]["query"])

print("\n" + "="*60)
print("Final Answer:")
print(result)


Final Answer:
Yes, the Cozy Comfort Pullover Set has side pockets. This information can be found in the "Additional Features" section of the document with metadata {'source': 'OutdoorClothingCatalog_1000.csv', 'row': 10}.


In [17]:
# Cell 16: Disable Debug Mode

# Turn off debug mode for cleaner output
langchain.debug = False

print("✅ Debug mode disabled")


✅ Debug mode disabled


In [18]:
# Cell 19: Generate Predictions for All Examples

# Run QA chain on all examples to get predictions
predictions = [{"query": ex["query"], "answer": ex["answer"], "result": qa.invoke(ex["query"])} for ex in examples]

print(f"✅ Generated predictions for {len(predictions)} examples")

✅ Generated predictions for 7 examples


## LLM-Assisted Evaluation

**The Challenge:** How do we automatically evaluate if answers are correct?
- Can't use exact string matching (too strict)
- Answers can be phrased differently but mean the same thing
- Need semantic comparison

**The Solution:** Use an LLM to grade the answers!


In [22]:
# Cell 21: Create Evaluation Chain

from langchain_core.prompts import ChatPromptTemplate

# Create evaluation prompt
eval_prompt = ChatPromptTemplate.from_template("""Evaluate if the predicted answer correctly answers the question based on the expected answer.

Question: {query}
Expected Answer: {answer}
Predicted Answer: {result}

Respond with only 'CORRECT' or 'INCORRECT'.""")

# Create evaluation chain
eval_chain = eval_prompt | llm | StrOutputParser()

print("✅ Evaluation chain created")

✅ Evaluation chain created


In [23]:
# Cell 22: Evaluate Predictions

# Use LLM to grade the predictions
graded_outputs = [{"text": eval_chain.invoke({"query": pred["query"], "answer": pred["answer"], "result": pred["result"]})} for pred in predictions]

print("✅ Evaluation complete")

✅ Evaluation complete


In [24]:
# Cell 21: Inspect First Graded Output

print("Detailed view of first graded output:")
print(graded_outputs[0])


Detailed view of first graded output:
{'text': 'CORRECT'}


## Why LLM-Based Evaluation Works

Let's examine Example 0:
- **Question:** "Does the Cozy Comfort Pullover Set have side pockets?"
- **Expected:** "Yes"
- **Predicted:** "The Cozy Comfort Pullover Set, Stripe does have side pockets."

### String Matching Would Fail:
- Expected and predicted strings are completely different
- "Yes" doesn't appear in the predicted answer
- No regex would catch this

### LLM Evaluation Succeeds:
- Understands semantic meaning
- Recognizes both answers convey the same information
- Grades as CORRECT

This is why traditional evaluation metrics fail for LLMs and why we need LLM-assisted evaluation.


In [25]:
# Cell 23: Count Correct Predictions

# Count how many predictions were graded as correct
correct_count = sum(
    1 for output in graded_outputs 
    if 'CORRECT' in output['text'].upper()
)

print(f"Evaluation Summary:")
print(f"  Total examples: {len(examples)}")
print(f"  Correct predictions: {correct_count}")
print(f"  Accuracy: {correct_count/len(examples)*100:.1f}%")


Evaluation Summary:
  Total examples: 7
  Correct predictions: 7
  Accuracy: 100.0%


In [26]:
# Cell 26: LangChain Evaluation Platform

print("""## LangChain Evaluation Platform

While this notebook shows evaluation in code, LangChain also provides a **UI-based evaluation platform** that:

### Features:
- **Persistent Sessions** - Track all runs over time
- **Visual Debugging** - See chain steps in a tree view
- **Dataset Creation** - Save examples for future evaluation
- **Comparison** - Compare different chain configurations
- **Collaboration** - Share results with team

### How to Use:
1. Sign up at https://smith.langchain.com/
2. Add `LANGCHAIN_TRACING_V2=true` to your .env
3. Add your API key to .env
4. Run chains normally - they'll auto-log to the platform

### Benefits Over Notebook:
- Historical tracking
- Better visualization
- Team collaboration
- Production monitoring
""")

## LangChain Evaluation Platform

While this notebook shows evaluation in code, LangChain also provides a **UI-based evaluation platform** that:

### Features:
- **Persistent Sessions** - Track all runs over time
- **Visual Debugging** - See chain steps in a tree view
- **Dataset Creation** - Save examples for future evaluation
- **Comparison** - Compare different chain configurations
- **Collaboration** - Share results with team

### How to Use:
1. Sign up at https://smith.langchain.com/
2. Add `LANGCHAIN_TRACING_V2=true` to your .env
3. Add your API key to .env
4. Run chains normally - they'll auto-log to the platform

### Benefits Over Notebook:
- Historical tracking
- Better visualization
- Team collaboration
- Production monitoring



In [27]:
# Cell 25: Export Results for Analysis

import json

# Create evaluation report
evaluation_report = {
    "model": llm_model,
    "total_examples": len(examples),
    "predictions": [
        {
            "question": pred['query'],
            "expected": pred['answer'],
            "predicted": pred['result'],
            "grade": graded_outputs[i]['text']
        }
        for i, pred in enumerate(predictions)
    ]
}

# Save to file
with open('evaluation_report.json', 'w') as f:
    json.dump(evaluation_report, f, indent=2)

print("✅ Evaluation report saved to 'evaluation_report.json'")


✅ Evaluation report saved to 'evaluation_report.json'


## Summary: Evaluation Best Practices

### 1. **Create Diverse Test Sets**
- Mix manual and LLM-generated examples
- Cover edge cases and common queries
- Include expected failures

### 2. **Use Debug Mode During Development**
- Understand what's being passed to the LLM
- Check retrieved documents for relevance
- Monitor token usage for cost control

### 3. **Automate Evaluation with LLMs**
- String matching fails for open-ended generation
- LLMs can judge semantic similarity
- Fast iteration on improvements

### 4. **Track Over Time**
- Save evaluation results
- Compare before/after changes
- Use evaluation platform for production monitoring

### Common Pitfalls:
❌ Not enough test examples (need 20-100+)  
❌ Only testing happy path  
❌ Ignoring retrieval quality (checking only final answer)  
❌ Not tracking changes over time  

### Evaluation Workflow:
- Create examples (manual + LLM)
- Run predictions through chain
- Grade with LLM evaluator
- Debug failures with langchain.debug=True
- Iterate and improve
- Repeat